<a href="https://colab.research.google.com/github/micko112/LLM-Chatbot---Rent-Rent-Out/blob/main/LLM_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q langchain langchain-openai langchain-chroma tiktoken langchain-community


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 k

In [3]:
import os
from google.colab import userdata
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [4]:
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [5]:
platform_rules = """
1. Pravo na postavljanje oglasa imaju isključivo lica koja su punoletna (18+ godina) i koja su prošla email verifikaciju.
2. Korisnici koji su prošli identifikaciju (KYC — upload ličnog dokumenta) dobijaju oznaku "Verifikovan" na profilu i imaju veće poverenje među drugim korisnicima.
3. Postavljanje oglasa je besplatno. Promotivne opcije se naplaćuju u kreditima: FEATURED 500 RSD (7 dana, vrh pretrage), PRIORITY 250 RSD (3 dana), HIGHLIGHTED 100 RSD (30 dana, vizuelni badge).
4. Dopuna kredita vrši se uplatom na zvanični račun platforme (Dimitrije Mitic, 265-0000006785327-58) uz obavezno korišćenje jedinstvenog poziva na broj koji se generiše na stranici `/credit`.
5. Krediti nisu povratni nakon što su iskorišćeni za aktivaciju promocije. Ako promocija nije konzumirana, korisnik može tražiti povraćaj u roku od 7 dana na mejl podrške.
6. Depozit koji zakupac eventualno uplaćuje zakupodavcu obavezno se vraća po isteku ugovora, ukoliko predmet iznajmljivanja nije oštećen i ukoliko je vraćen u dogovorenom roku.
7. Stanje iznajmljene stvari (fotografije, funkcionalnost) obavezno se dokumentuje pri primopredaji — i preporučuje se razmena tih fotografija u chat-u platforme kao dokaz.
8. Ukoliko stanodavac (vlasnik stvari) otkaže već prihvaćen ugovor bez opravdanog razloga, zakupac ima pravo na prioritetnu podršku i eventualnu naknadu troškova dokazanih kroz chat istoriju.
9. Otkazni rok za već prihvaćen ugovor je minimum 48 sati pre datuma početka — kasnije otkazivanje može rezultirati negativnom recenzijom i suspenzijom naloga kod ponavljanja.
10. Oštećenje iznajmljene stvari mora biti prijavljeno u roku od 48 sati od primopredaje. Prijava se vrši kroz chat platforme i mejlom na izdajemiznajmljujem.rs@gmail.com sa fotografijama.
11. Platforma ne posreduje u plaćanju naknade za rental (iznajmljivanje) — to se dogovara direktno između korisnika. Platforma naplaćuje isključivo promotivne opcije putem sistema kredita.
12. Zabranjeno je oglašavanje: oružja, narkotika, životinja, ljudskih organa, krađene robe, falsifikata, pirotehnike, alkohola licima mlađim od 18 godina, kao i bilo kakvih nelegalnih usluga.
13. Korisnik ima pravo na brisanje naloga i svih ličnih podataka u skladu sa Zakonom o zaštiti podataka o ličnosti (ZZPL) i GDPR-om — zahtev se šalje na izdajemiznajmljujem.rs@gmail.com.
14. Kasno vraćanje iznajmljene stvari može rezultirati dodatnom naknadom za svaki započet dan kašnjenja prema ceni iz originalnog oglasa.
15. Sporovi oko ugovora rešavaju se najpre direktno kroz chat platforme. Ukoliko korisnici ne postignu dogovor, slučaj se eskaliraju na izdajemiznajmljujem.rs@gmail.com — admini posreduju u roku od 3 radna dana.
16. Prijava sumnje na prevaru (traženje uplate na privatne račune van /credit stranice, lažni oglasi, pretnje) mora biti bezuslovna — suspenzija naloga je trenutna, a slučaj se predaje nadležnim organima ako postoje elementi krivičnog dela.
17. Recenzije se mogu ostaviti isključivo nakon završenog ugovora, u roku od 30 dana. Lažne recenzije, međusobno "nameštanje" i vređanja se brišu bez upozorenja.
18. Prava potrošača u skladu sa Zakonom o zaštiti potrošača — pravo na reklamaciju neispravne usluge, pravo na informisanost, pravo na zaštitu od nepoštene poslovne prakse — primenjuju se na sve komercijalne transakcije preko platforme.
"""

In [6]:
with open("pravila_izdavanja.txt", "w", encoding = "utf-8") as f:
  f.write(platform_rules)

In [7]:
loader = TextLoader("pravila_izdavanja.txt")
document = loader.load()
print("Successfuly loaded file")

Successfuly loaded file


CHUNKING

In [8]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 150,
    chunk_overlap = 20
)
splitted_docs = text_splitter.split_documents(document)
print(f"Splitted on {len(splitted_docs)} chunks")

Splitted on 34 chunks


SBERT embedding




In [9]:
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

vectors = Chroma.from_documents(
    documents = splitted_docs,
    embedding=embeddings_model
)

In [10]:
retriever = vectors.as_retriever(search_kwargs={"k": 2})

In [11]:
llm = ChatOpenAI(model = "gpt-4o-mini", temperature=0.6)

prompt_template = PromptTemplate(
    template = """Ti si gotivni ljubazni dobri agent za korisničku podršku sajta 'Izdajem Iznajmljujem'.
Na osnovu ponuđenog KONTEKSTA, odgovori na PITANJE korisnika.
Ako u KONTEKSTU nema odgovora, reci: "Nažalost, nemam tu informaciju u pravilniku." Nemoj nikada da izmišljaš pravila.

KONTEKST:
{context}

PITANJE KORISNIKA: {question}

ODGOVOR:""",
    input_variables=["context", "question"]
)

In [12]:
def format_docs(docs):
  return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()

)

In [13]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

In [14]:
# question = "Da li se placa  za oglas na vašem sajtu?"
# print(f"Korisnik pita: {question}")

# answer = rag_chain.invoke(question)
# print(f"Bot odgovara:\n{answer}")

Graf u LangGraph funkcionise tako sto svaki node (cvor) prima trenutno stanje koje obradjuje i azurira zatim prosledjuje sledecem cvoru. AgentState definise koje se informacije prenose u state-u.
AgentState pruza jednostavan nacin da se sacuva i upravlja state-om AI agentom u realnom vremenu. Informacije koje prenosimo su: question, dokumenats, is_relevatn i asnwer.

In [16]:
class AgentState(TypedDict):
    question: str
    documents: list[str]
    is_relevant: str
    answer: str

`retrieve_node` je izvrsni cvor koji je zaduzen za pronalazenje relevantnih informacija. Zadatak mu je da pomocu `retriever` objekta komunicira sa vektroskom bazom (Chroma).

Pitanje se konvertuje u vektor
(embedding) i pronalazi 2 (k=2) najslicnija pasusa iz Chroma koristeci kosinusnu slicnost.

Iteracija kroz dokument i izvlacenje cistog teksta (page_content) u novu listu stringova

In [17]:
def retrieve_node(state):
  docs = retriever.invoke(state["question"])
  doc_strings = [doc.page_content for doc in docs]
  return {"documents": doc_strings}

`grade_node` je evaluacioni cvor koji implementira napredni koncept Self-RAG. U ovom cvoru LLM se koristi iskljucivo za proveru cinjenica

Njegov zadatak je da uporedi *question* iz state-a sa dokumentima dobijenim iz `retrieve_node` i proceni da li dokumenti sadrze bar delimican odgovor na dobijeno pitanje. Cvor prihvata trenutno stanje - *state*

1.   Izvlaci pitanje iz *state*
2.   Spaja listu dobijenih dokumenta u jedan string - *context*
3.   Kreira se prompt koji ima binaran izlaz
4.   Poziva se LLM da izvrsi prompt, ako tekst sadrzi odgovor na pitanje, postavlja da je pitanje relevantno i obrnuto.






In [18]:
def grade_node(state):
  question = state["question"]
  context = "\n".join(state["documents"])

  prompt = f"""Proveri da li TEKST sadrži odgovor na PITANJE.
    Odgovori ISKLJUČIVO sa: yes ili no.
    TEKST: {context}
    PITANJE: {question}"""

  decision = llm.invoke(prompt).content.strip().lower()

  if "yes" in decision:
    return {"is_relevant": "yes"}
  else:
    return {"is_relevant": "no"}

`generate_node` je izvršni cvor odgovoran za sintezu konacnog teksta, baziranu isključivo na kontekstu obezbeđenom putem RAG (Retrieval-Augmented Generation) metode.

In [19]:
def generate_node(state):
  context = "\n\n".join(state["documents"])
  question = state["question"]

  prompt = f"""Ti si gotivan ljubazan asistent na sajtu Izdajem Iznajmljujem.
    Na osnovu KONTEKSTA, odgovori na PITANJE.
    KONTEKST: {context}
    PITANJE: {question}"""

  answer = llm.invoke(prompt).content
  return {"answer": answer}

`escalate_node` je Fallback mehanizam. U pitanju je deterministicki cvor koji vraca unapred definisani tekstualni odgovor (ne poziva LLM).

In [20]:
def escalate_node(state):
    return {"answer": "Nažalost, nemam tu informaciju u pravilniku. Da li želiš da te povežem sa operaterom?"}

`chat_node` sluzi kao direktan kanal (bypass) ka osnovnom jezickom modelu, zaduzen je za obradu opstih upita koji nisu vezani za domen.

Primer:

*   question: "Kako si?"
*   answer: "Super sam, kako si ti?"

Kada router detektuje da pitanje nije tehnicko ili nije vezano za koriscenje platforme, sistem preskace Chroma - vektorsku bazu i dozvoljava modelu da slobodno komunicira sa korisnikom, koristeci svoje predtrenirano znanje.



In [21]:
def chat_node(state):
  question = state["question"]
  prompt = f"""Ti si gotivan ljubazan i dobar asistent na sajtu Izdajem Iznajmljujem.
  Korisnik ti kaže: '{question}'. Odgovori kratko i prijateljski."""
  answer = llm.invoke(prompt).content
  return {"answer": answer}

Router je funkcija za uslovno usmeravanje unutar grafa. Zadatak je da klasifikuje namere korisnika i usmeravanje na sledeci cvor. Sprecava nepotrebno pretrazivanje vektorske baze (RAG) ukoliko se pitanje ne tice domena platforme. Time ustedi tokene i vreme.  
*   Cita question (pitanje iz state-a)
*   Kreira prompt za LLM koji binarno odlucuje da li se odnosi na domen platforme ili ne
*   Ako vrati "DA", preusmerava graf na retrieve_node, odnosno na pretragu dokumentacije
*   Ako vrati "NE", preusmerava graf na chat_node koji sluzi za obradu opstih upita (small talk / out of domain queries)





In [22]:
def router(state):
  question = state["question"]
  prompt = f"""Da li se ovo pitanje odnosi na pravila, oglase, depozite, plaćanje, kredite ili funkcionisanje sajta za nekretnine?
  Pitanje: {question}
  Odgovori ISKLJUČIVO sa DA ili NE."""

  decision = llm.invoke(prompt).content.strip().lower()

  if "da" in decision:
    return "retrieve_node"
  else:
    return "chat_node"

`check_relevance` je staticka funkcija (ne koristi LLM) za uslovno usmeravanje (conditional edge). Ona donosi konacnu odluku o relevantnosti na osnovu ocene koju je dodelio `grade_node`.

In [23]:
def check_relevance(state):
    if state["is_relevant"] == "yes":
        return "generate_node"
    else:
        return "escalate_node"

In [51]:
workflow = StateGraph(AgentState)

workflow.add_node("retrieve_node", retrieve_node)
workflow.add_node("grade_node", grade_node)
workflow.add_node("generate_node", generate_node)
workflow.add_node("escalate_node", escalate_node)
workflow.add_node("chat_node", chat_node)

In [52]:
workflow.add_conditional_edges(
    START,
    router,
    {
        "retrieve_node": "retrieve_node",
        "chat_node": "chat_node"
    }
)

In [53]:
workflow.add_edge("retrieve_node", "grade_node")

In [54]:
workflow.add_conditional_edges(
    "grade_node",
    check_relevance,
    {
        "generate_node": "generate_node",
        "escalate_node": "escalate_node"
    }
)

In [55]:
workflow.add_edge("generate_node", END)
workflow.add_edge("escalate_node", END)
workflow.add_edge("chat_node", END)

In [56]:
agent_app = workflow.compile()
print("Graf je uspesno kompajliran na svim cvorovima")

Graf je uspesno kompajliran na svim cvorovima


In [57]:
question_1 = "Da li se placa za oglas na vašem sajtu?"
print(f"KORISNIK: {question_1}")
result_1 = agent_app.invoke({"question": question_1})
print(f"BOT: {result_1['answer']}\n")
print("-" * 50)

# Test 2: Pitanje za običan razgovor (Testiramo skretničara!)
question_2 = "Da li znas kvadratnu jednacinu?"
print(f"\nKORISNIK: {question_2}")
result_2 = agent_app.invoke({"question": question_2})
print(f"BOT: {result_2['answer']}")

KORISNIK: Da li se placa za oglas na vašem sajtu?
BOT: Nažalost, nemam tu informaciju u pravilniku. Da li želiš da te povežem sa operaterom?

--------------------------------------------------

KORISNIK: Da li znas kvadratnu jednacinu?
BOT: Da, znam kvadratnu jednčinu! To je oblik \(ax^2 + bx + c = 0\). Ako ti treba pomoć oko rešavanja, slobodno pitaj! 😊


In [58]:
print("\n--- TEST 1: Pitanje koje postoji u pravilniku (RAG -> GRADE -> GENERATE) ---")
q1 = "Koliko dana traje PRIORITY oglas i koliko košta?"
print(f"KORISNIK: {q1}")
print(f"BOT: {agent_app.invoke({'question': q1})['answer']}")

print("\n--- TEST 2: Pitanje o sajtu, ali NE POSTOJI u pravilniku (RAG -> GRADE -> ESCALATE) ---")
q2 = "Kako da promenim temu sajta u crnu?"
print(f"KORISNIK: {q2}")
print(f"BOT: {agent_app.invoke({'question': q2})['answer']}")

print("\n--- TEST 3: Običan razgovor (START -> CHAT) ---")
q3 = "Da li znas kvadratnu jednacinu?"
print(f"KORISNIK: {q3}")
print(f"BOT: {agent_app.invoke({'question': q3})['answer']}")


--- TEST 1: Pitanje koje postoji u pravilniku (RAG -> GRADE -> GENERATE) ---
KORISNIK: Koliko dana traje PRIORITY oglas i koliko košta?
BOT: PRIORITY oglas traje 3 dana i košta 250 RSD.

--- TEST 2: Pitanje o sajtu, ali NE POSTOJI u pravilniku (RAG -> GRADE -> ESCALATE) ---
KORISNIK: Kako da promenim temu sajta u crnu?
BOT: Nažalost, nemam tu informaciju u pravilniku. Da li želiš da te povežem sa operaterom?

--- TEST 3: Običan razgovor (START -> CHAT) ---
KORISNIK: Da li znas kvadratnu jednacinu?
BOT: Da, znam! Kvadratna jednačina je oblika ax² + bx + c = 0, gde su a, b i c koeficijenti. Ako ti treba pomoć oko rešenja, slobodno pitaj! 😊
